In [3]:
import os 
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\IMDB Projetc'

In [4]:
# os.chdir('../')
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\IMDB Projetc'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir              : Path
    model_path            : Path
    history_path          : Path
    test_data_path        : Path
    report_path           : Path
    confusion_matrix_path : Path
    accuracy_plot_path    : Path
    loss_plot_path        : Path


@dataclass(frozen=True)
class Params:
    max_words     : int
    max_len       : int
    test_size     : float
    random_state  : int
    batch_size    : int
    epochs        : int
    embedding_dim : int
    lstm_units    : int
    dropout_rate  : float
    learning_rate : float

In [6]:
from src.IMDBSentimentAnalysis.constants import *
from src.IMDBSentimentAnalysis.utils.main import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])


    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories([config.root_dir])
        return ModelEvaluationConfig(
            root_dir              = config.root_dir,
            model_path            = config.model_path,
            history_path          = config.history_path,
            test_data_path        = config.test_data_path,
            report_path           = config.report_path,
            confusion_matrix_path = config.confusion_matrix_path,
            accuracy_plot_path    = config.accuracy_plot_path,
            loss_plot_path        = config.loss_plot_path
        )


    def get_params(self) -> Params:
        params = self.params
        return Params(
            max_words     = params.max_words,
            max_len       = params.max_len,
            test_size     = params.test_size,
            random_state  = params.random_state,
            batch_size    = params.batch_size,
            epochs        = params.epochs,
            embedding_dim = params.embedding_dim,
            lstm_units    = params.lstm_units,
            dropout_rate  = params.dropout_rate,
            learning_rate = params.learning_rate
        )

[ 2026-05-18 23:37:06,256 : INFO : __init__ : Logger initialized successfully ]


In [7]:
import json
import pickle
import numpy as np
import logging
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

from tensorflow.keras.models import load_model

# from src.IMDBSentimentAnalysis.entity import ModelEvaluationConfig

logger = logging.getLogger(__name__)


class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config


    # ── Step 1: Load Model ────────────────────────────────────────
    def load_model(self):
        try:
            model = load_model(self.config.model_path)
            logger.info(f"Model loaded      : {self.config.model_path}")
            return model

        except Exception as e:
            logger.error(f"Model load failed : {e}")
            raise e


    # ── Step 2: Load Test Data ────────────────────────────────────
    def load_test_data(self):
        try:
            test       = np.load(self.config.test_data_path)
            X_test     = test['X']
            y_test     = test['y']

            logger.info(f" Test data loaded  : {X_test.shape}")
            return X_test, y_test

        except Exception as e:
            logger.error(f"Test data load failed: {e}")
            raise e


    # ── Step 3: Load History ──────────────────────────────────────
    def load_history(self):
        try:
            with open(self.config.history_path, 'rb') as f:
                history = pickle.load(f)

            logger.info(f"History loaded    : {self.config.history_path}")
            return history

        except Exception as e:
            logger.error(f" History load failed: {e}")
            raise e


    # ── Step 4: Generate Predictions ─────────────────────────────
    def predict(self, model, X_test):
        try:
            y_prob = model.predict(X_test, verbose=0)
            y_pred = (y_prob >= 0.5).astype(int).flatten()

            logger.info(" Predictions generated")
            return y_pred, y_prob

        except Exception as e:
            logger.error(f" Prediction failed: {e}")
            raise e


    # ── Step 5: Generate & Save Report ───────────────────────────
    def save_report(self, y_test, y_pred):
        try:
            report = {
                "accuracy"  : np.round(accuracy_score(y_test, y_pred), 4),
                "precision" : np.round(precision_score(y_test, y_pred), 4),
                "recall"    : np.round(recall_score(y_test, y_pred), 4),
                "f1_score"  : np.round(f1_score(y_test, y_pred), 4),
            }

            with open(self.config.report_path, 'w') as f:
                json.dump(report, f, indent=4)

            logger.info(f" Evaluation Report:")
            logger.info(f"   Accuracy  : {report['accuracy']  * 100:.2f}%")
            logger.info(f"   Precision : {report['precision'] * 100:.2f}%")
            logger.info(f"   Recall    : {report['recall']    * 100:.2f}%")
            logger.info(f"   F1 Score  : {report['f1_score']  * 100:.2f}%")
            logger.info(f"\n{classification_report(y_test, y_pred, target_names=['Negative', 'Positive'])}")

        except Exception as e:
            logger.error(f" Report save failed: {e}")
            raise e


    # ── Step 6: Confusion Matrix Plot ─────────────────────────────
    def plot_confusion_matrix(self, y_test, y_pred):
        try:
            cm = confusion_matrix(y_test, y_pred)

            plt.figure(figsize=(8, 6))
            sns.heatmap(
                cm,
                annot      = True,
                fmt        = 'd',
                cmap       = 'Blues',
                xticklabels= ['Negative', 'Positive'],
                yticklabels= ['Negative', 'Positive']
            )
            plt.title('Confusion Matrix', fontsize=15, fontweight='bold')
            plt.xlabel('Predicted Label', fontsize=12)
            plt.ylabel('True Label', fontsize=12)
            plt.tight_layout()
            plt.savefig(self.config.confusion_matrix_path, dpi=150)
            plt.close()

            logger.info(f" Confusion matrix saved: {self.config.confusion_matrix_path}")

        except Exception as e:
            logger.error(f" Confusion matrix plot failed: {e}")
            raise e


    # ── Step 7: Accuracy Plot ─────────────────────────────────────
    def plot_accuracy(self, history: dict):
        try:
            plt.figure(figsize=(10, 5))
            plt.plot(history['accuracy'],     label='Train Accuracy', color='blue',   linewidth=2)
            plt.plot(history['val_accuracy'], label='Val Accuracy',   color='orange', linewidth=2)
            plt.title('Model Accuracy over Epochs', fontsize=15, fontweight='bold')
            plt.xlabel('Epoch', fontsize=12)
            plt.ylabel('Accuracy', fontsize=12)
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(self.config.accuracy_plot_path, dpi=150)
            plt.close()

            logger.info(f" Accuracy plot saved   : {self.config.accuracy_plot_path}")

        except Exception as e:
            logger.error(f" Accuracy plot failed: {e}")
            raise e


    # ── Step 8: Loss Plot ─────────────────────────────────────────
    def plot_loss(self, history: dict):
        try:
            plt.figure(figsize=(10, 5))
            plt.plot(history['loss'],     label='Train Loss', color='blue',   linewidth=2)
            plt.plot(history['val_loss'], label='Val Loss',   color='red',    linewidth=2)
            plt.title('Model Loss over Epochs', fontsize=15, fontweight='bold')
            plt.xlabel('Epoch', fontsize=12)
            plt.ylabel('Loss', fontsize=12)
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(self.config.loss_plot_path, dpi=150)
            plt.close()

            logger.info(f" Loss plot saved       : {self.config.loss_plot_path}")

        except Exception as e:
            logger.error(f" Loss plot failed: {e}")
            raise e


    # ── Master Method ─────────────────────────────────────────────
    def run(self):
        model          = self.load_model()
        X_test, y_test = self.load_test_data()
        history        = self.load_history()
        y_pred, y_prob = self.predict(model, X_test)

        self.save_report(y_test, y_pred)
        self.plot_confusion_matrix(y_test, y_pred)
        self.plot_accuracy(history)
        self.plot_loss(history)

In [8]:
import logging
# from src.IMDBSentimentAnalysis.config.configuration import ConfigurationManager
# from src.IMDBSentimentAnalysis.components.model_evaluation import ModelEvaluation

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

STAGE_NAME = "Model Evaluation Stage"


class ModelEvaluationPipeline:
    def __init__(self):
        pass

    def run(self):
        config      = ConfigurationManager()
        eval_cfg    = config.get_model_evaluation_config()

        evaluation  = ModelEvaluation(config=eval_cfg)
        evaluation.run()


if __name__ == "__main__":
    try:
        logger.info(f">>>>>> {STAGE_NAME} started <<<<<<")
        pipeline = ModelEvaluationPipeline()
        pipeline.run()
        logger.info(f">>>>>> {STAGE_NAME} completed <<<<<<\n\nx==========x")

    except Exception as e:
        logger.exception(e)
        raise e

[ 2026-05-18 23:37:15,003 : INFO : 69252550 : >>>>>> Model Evaluation Stage started <<<<<< ]
[ 2026-05-18 23:37:15,012 : INFO : main : YAML file: config\config.yaml Loaded Successfully ]
[ 2026-05-18 23:37:15,017 : INFO : main : YAML file: params.yaml Loaded Successfully ]
[ 2026-05-18 23:37:15,017 : INFO : main : Create Directory at: artifacts ]
[ 2026-05-18 23:37:15,017 : INFO : main : Create Directory at: artifacts/model_evaluation ]
[ 2026-05-18 23:37:15,234 : WARNING : config : TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin. ]
[ 2026-05-18 23:37:15,234 : WARNING : saving_utils : Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model. ]
[ 2026-05-18 23:37:15,239 : INFO : 1724862079 : Model loaded      : artifacts/model_trainer/model.h5 ]
[ 2026-05-18